# 🛡️ Multi-Class Network Intrusion Detection System (NIDS) & Zero-Day Anomaly Detection
**Author:** Arjuna Fransesco  
**Domain:** Cybersecurity / SecOps & Applied Machine Learning  
**Target:** Automated Detection and Triage of DoS/DDoS, PortScan, Brute Force, and Botnet C2 Network Attacks

---
## 📌 1. Project Overview & SecOps Objective
Modern Security Operations Centers (SOCs) analyze gigabytes of network telemetry per second. Traditional signature-based IDS fails to detect evasive zero-day attacks and polymorphic traffic.

This project implements a dual-stage **Machine Learning Network Intrusion Detection System (NIDS)**:
1. **Supervised Multi-Class Threat Classifier** (Voting Ensemble: LightGBM + XGBoost + Random Forest) categorizing known attack vectors (`Benign`, `DoS_DDoS`, `PortScan`, `BruteForce`, `Botnet_C2`).
2. **Unsupervised Anomaly Detector** (Isolation Forest) flagging novel behavioral deviations and zero-day threats.
3. High-standard SecOps metrics: Maximizing **Attack Detection Rate** while keeping **False Alarm Rate (FAR) < 1%**.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

sys.path.append('..')
from src.data_loader import load_dataset, split_features_targets
from src.feature_engineering import build_preprocessor
from src.models import get_supervised_classifiers, build_voting_threat_detector, build_zero_day_anomaly_detector
from src.evaluate import calculate_nids_metrics, benchmark_classifiers

print("[+] SecOps Environment and NIDS modules loaded successfully!")

## 📊 2. Network Flow Telemetry Ingestion & Exploratory Analysis

In [2]:
df = load_dataset('../data/network_traffic_dataset.csv')
print(f"Telemetry Records: {df.shape[0]:,} flows × {df.shape[1]} telemetry attributes")
df.head()

In [3]:
# Threat distribution
fig, ax = plt.subplots(figsize=(8, 4))
threat_counts = df['attack_category'].value_counts()
colors = ['#10B981', '#EF4444', '#F59E0B', '#8B5CF6', '#EC4899']
threat_counts.plot(kind='bar', color=colors, ax=ax)
plt.title('Network Traffic Attack Category Distribution')
plt.xlabel('Traffic Classification')
plt.ylabel('Flow Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [4]:
# TCP SYN Count vs ACK Count across categories
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df.sample(min(1500, len(df)), random_state=42),
    x='syn_count', y='ack_count',
    hue='attack_category',
    palette='tab10',
    alpha=0.7
)
plt.title('TCP SYN vs ACK Flag Dynamics by Attack Category')
plt.xlabel('SYN Flag Count')
plt.ylabel('ACK Flag Count')
plt.xscale('log')
plt.yscale('log')
plt.tight_layout()
plt.show()

## 🛠️ 3. Feature Engineering & Telemetry Transformations

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X, y_cat, y_bin = split_features_targets(df)
le = LabelEncoder()
y_encoded = le.fit_transform(y_cat)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42
)

preprocessor = build_preprocessor()
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f"Transformed Feature Matrix Shape: {X_train_trans.shape}")
print(f"Attack Classes ({len(le.classes_)}): {list(le.classes_)}")

## 🚀 4. Multi-Class Classifier Benchmarking (5-Fold Stratified CV)

In [6]:
candidate_models = get_supervised_classifiers(random_state=42)
benchmark_df = benchmark_classifiers(candidate_models, X_train_trans, y_train, cv_folds=5, random_state=42)
benchmark_df

## 🏆 5. Final Voting Ensemble Training & Test Evaluation

In [7]:
# Train soft-voting threat detector
voting_clf = build_voting_threat_detector(random_state=42)
voting_clf.fit(X_train_trans, y_train)

# Evaluate on holdout test set
y_test_pred = voting_clf.predict(X_test_trans)
test_metrics = calculate_nids_metrics(y_test, y_test_pred, label_encoder=le)

print("=== NIDS Holdout Test Metrics ===")
for k, v in test_metrics.items():
    if k != 'Confusion_Matrix':
        print(f"{k:<25}: {v}")

In [8]:
# Multi-Class Confusion Matrix Heatmap
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Multi-Class NIDS Confusion Matrix (Holdout Test Set)')
plt.xlabel('Predicted Threat Class')
plt.ylabel('Actual Threat Class')
plt.tight_layout()
plt.show()

## 🕵️ 6. Unsupervised Zero-Day Anomaly Detection

In [9]:
# Isolation Forest trained strictly on benign baseline traffic
benign_idx = list(le.classes_).index('Benign')
iso_forest = build_zero_day_anomaly_detector(contamination=0.04, random_state=42)
iso_forest.fit(X_train_trans[y_train == benign_idx])

test_anomaly_scores = iso_forest.decision_function(X_test_trans)

plt.figure(figsize=(10, 5))
for cls_name in le.classes_:
    cls_idx = list(le.classes_).index(cls_name)
    scores = test_anomaly_scores[y_test == cls_idx]
    sns.kdeplot(scores, label=cls_name, fill=True, alpha=0.2)

plt.axvline(0, color='red', linestyle='--', label='Anomaly Decision Threshold')
plt.title('Zero-Day Isolation Forest Anomaly Score Distribution')
plt.xlabel('Anomaly Score (Lower = More Anomalous)')
plt.legend()
plt.tight_layout()
plt.show()

## 📌 7. Conclusion & SOC Deployment Guidelines
- **Voting Ensemble** achieved $> 99\%$ multi-class accuracy and $> 99\%$ detection rate across DoS, PortScan, BruteForce, and Botnet traffic.
- **Isolation Forest** successfully separates benign baseline flows from malicious outliers without requiring labeled zero-day signatures.
- Production deployment via **Streamlit SOC Dashboard (`app.py`)** with real-time MITRE ATT&CK tactical recommendations.